# Debugging a Kubernetes network latency spike between services using Elastic

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/kubernetes-network-latency-debugging). It runs the article's ES|QL queries against your own run of the lab.

1. Copy `.env.example` to `.env` and fill it in.
2. Run `k8s/lab.sh up`, then `k8s/lab.sh scenario` (about 80 minutes). The scenario records the incident times in `.env`.
3. Run the cells. The last one deletes the kind cluster.

Requires Elasticsearch 9.4+, Docker, kind, kubectl and Helm.

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import os
import warnings
from datetime import datetime, timedelta

from dotenv import load_dotenv
from elasticsearch import Elasticsearch, ElasticsearchWarning

warnings.filterwarnings("ignore", category=ElasticsearchWarning)
load_dotenv()
es = Elasticsearch(
    os.environ["ELASTICSEARCH_URL"], api_key=os.environ["ELASTICSEARCH_API_KEY"]
)
print(es.info()["version"]["number"])


def at(key, minutes=0):
    """Incident time written by k8s/lab.sh, rounded down to the minute and shifted."""
    t = datetime.fromisoformat(os.environ[key].replace("Z", "+00:00")).replace(second=0)
    return t + timedelta(minutes=minutes)


WINDOWS = {
    "baseline": (at("LAB_DEGRADED_START", -10), at("LAB_DEGRADED_START")),
    "degraded": (at("LAB_DEGRADED_START"), at("LAB_DEGRADED_END")),
    "noisy": (at("LAB_NOISY_START"), at("LAB_NOISY_END")),
    "dns": (at("LAB_DNS_START", 1), at("LAB_DNS_FIX")),
    "around_dns": (at("LAB_DNS_START", -5), at("LAB_DNS_FIX", 12)),
    "after_fix": (at("LAB_DNS_FIX", 2), at("LAB_DNS_FIX", 12)),
}


def esql(query, window):
    start, end = WINDOWS[window]
    time_range = {"gte": start.isoformat(), "lt": end.isoformat()}
    response = es.esql.query(query=query, filter={"range": {"@timestamp": time_range}})
    print(f"\n{window}: {start:%H:%M}-{end:%H:%M} UTC")
    print(" | ".join(column["name"] for column in response["columns"]))
    for row in response["values"]:
        print(" | ".join(str(value) for value in row))

## 2. Split request time with OpenTelemetry trace spans

In [ ]:
PHASES = """FROM traces-generic.otel-default
| WHERE service.name == "checkout"
    AND span.name IN ("HTTP GET", "http.dns", "http.connect")
| STATS calls = COUNT(*),
        p50_ms = ROUND(PERCENTILE(duration, 50) / 1000000.0, 1),
        p99_ms = ROUND(PERCENTILE(duration, 99) / 1000000.0, 1),
        errors = COUNT(*) WHERE status.code == "Error"
  BY span.name"""

esql(PHASES, "dns")
esql(PHASES, "baseline")

## 3. Find where DNS queries go missing with CoreDNS logs

In [ ]:
SLOW_FAST = """FROM traces-generic.otel-default
| WHERE service.name == "checkout" AND span.name == "http.dns"
| STATS slow_lookups = COUNT(*) WHERE duration > 1000000000,
        fast_lookups = COUNT(*) WHERE duration <= 1000000000
  BY minute = BUCKET(@timestamp, 1 minute)"""

COREDNS = r"""FROM logs-generic.otel-default
| WHERE k8s.deployment.name == "coredns"
| GROK body.text "\\[%{WORD:level}\\] %{IP:client_ip}:%{INT} - %{INT} \"%{WORD:qtype} IN %{NOTSPACE:qname} %{WORD:proto} %{INT} %{WORD} %{INT}\" %{WORD:rcode} %{NOTSPACE} %{INT} %{NUMBER:seconds:float}s"
| WHERE qname == "inventory.shop.svc.cluster.local."
| STATS answered = COUNT(*) BY minute = BUCKET(@timestamp, 1 minute), k8s.pod.name, k8s.node.name"""

EVENTS = """FROM logs-k8seventsreceiver.otel-default
| WHERE attributes.k8s.namespace.name == "kube-system"
    AND k8s.object.name LIKE "coredns*"
    AND k8s.event.reason == "Scheduled"
| KEEP @timestamp, k8s.event.reason, k8s.object.name, body.text
| SORT @timestamp"""

esql(SLOW_FAST + "\n| SORT minute", "around_dns")
esql(COREDNS + "\n| SORT minute", "dns")
esql(EVENTS, "around_dns")

## 4. Verify the network policy fix

In [ ]:
esql(PHASES, "after_fix")

## 5. Degraded node: every pod is slow, none is busy

In [ ]:
OUTSIDE = """FROM traces-generic.otel-default
| WHERE span.name IN ("HTTP GET", "GET /stock")
| STATS client = MAX(duration) WHERE span.name == "HTTP GET",
        server = MAX(duration) WHERE span.name == "GET /stock",
        inventory_node = MAX(k8s.node.name) WHERE span.name == "GET /stock"
  BY trace.id
| EVAL outside_server_ms = (client - server) / 1000000.0
| STATS requests = COUNT(*),
        p50_ms = ROUND(PERCENTILE(outside_server_ms, 50), 1),
        p95_ms = ROUND(PERCENTILE(outside_server_ms, 95), 1)
  BY inventory_node"""

TX = """TS metrics-kubeletstatsreceiver.otel-default
| WHERE k8s.node.name == "net-latency-worker3"
    AND direction == "transmit"
    AND k8s.namespace.name NOT IN ("kube-system", "opentelemetry-operator-system")
| STATS tx_mbit_s = ROUND(SUM(RATE(k8s.pod.network.io)) * 8 / 1000000, 2)
  BY k8s.namespace.name, k8s.pod.name
| SORT tx_mbit_s DESC NULLS LAST"""

esql(OUTSIDE, "degraded")
esql(TX, "degraded")

## 6. Noisy-neighbor pod: one pod fills the uplink

In [ ]:
esql(OUTSIDE, "noisy")
esql(TX, "noisy")

## 7. Clean up

In [ ]:
!k8s/lab.sh down